# A Simple Symbolic Calculator

F# counterpart of `Python/Chapter-10/Symbolic-Calculator.ipynb`. Build once with `python FSharp/tools/build.py`, then run this notebook from top to bottom with a fresh .NET (F#) kernel. The scanner and parser are generated from real FsLexYacc specifications.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "Lalr/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "Lalr/bin/Debug/net9.0/Lalr.dll"
#load "../Support/Visualization.fsx"
open System
open System.IO
open FormalLanguages.Lalr
open Visualization

## Typed syntax and scanner

The semantic actions construct a closed expression type. Numeric tokens are converted once, in the scanner; arithmetic entry points require integer literals, and the symbolic calculator accepts decimal and exponent forms; punctuation becomes a token constructor. Newlines advance the source position, and illegal input raises a positioned error.

```fsharp
namespace FormalLanguages.Lalr

/// A closed syntax tree makes every interpreter case visible to the compiler.
type Expression =
    | Number of float
    | Name of string
    | Unary of string * Expression
    | Binary of string * Expression * Expression

type Statement =
    | Assign of string * Expression
    | Evaluate of Expression

type Lexeme =
    | Numeric of float
    | Identifier of string
    | Operator of string
    | Character of char
    | End
```

The shared `.fsl` file supplies an expression mode and a character mode:

```text
{
module FormalLanguages.Lalr.Lexer
open System.Globalization
open FSharp.Text.Lexing
}
let digit = ['0'-'9']
let letter = ['a'-'z' 'A'-'Z']
let number = ('0' | ['1'-'9'] digit*) ('.' digit+)? (['e' 'E'] ['+' '-']? digit+)?
rule token = parse
| [' ' '\t' '\r']+ { token lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
| number { Numeric (System.Double.Parse(LexBuffer<char>.LexemeString lexbuf, CultureInfo.InvariantCulture)) }
| letter (letter | digit | ['_' '<' '>'])* { Identifier (LexBuffer<char>.LexemeString lexbuf) }
| ":=" { Operator ":=" }
| ['+' '-' '*' '/' '^' '(' ')' ';' '↔' '→' '∧' '∨' '¬' '⊤' '⊥'] { Operator (LexBuffer<char>.LexemeString lexbuf) }
| eof { End }
| _ { failwithf "Illegal character %A at line %d, column %d" (LexBuffer<char>.LexemeString lexbuf) lexbuf.StartPos.Line (lexbuf.StartPos.Column + 1) }
and character = parse
| [' ' '\t' '\r']+ { character lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; character lexbuf }
| ['U' 'V' 'W' 'X' 'v' 'w' 'x' 'y' 'z'] { Character (lexbuf.LexemeChar 0) }
| eof { End }
| _ { failwithf "Illegal character %A" (LexBuffer<char>.LexemeString lexbuf) }
```

## Parser specification

FsYacc generates token constructors and parse tables from this `.fsy` specification. `$1`, `$2`, and `$3` refer to values on the right-hand side. `main` consumes `EOF`, so a valid prefix with trailing junk is rejected.

```text
%{
open FormalLanguages.Lalr
%}
%token <float> NUMBER
%token <string> ID
%token ASSIGN PLUS MINUS TIMES DIVIDE LPAREN RPAREN SEMICOLON EOF
%start main
%type <FormalLanguages.Lalr.Statement> main
%%
main: statement EOF { $1 }
statement:
  | ID ASSIGN expr SEMICOLON { Assign ($1, $3) }
  | expr SEMICOLON { Evaluate $1 }
expr:
  | expr PLUS product { Binary ("+", $1, $3) }
  | expr MINUS product { Binary ("-", $1, $3) }
  | product { $1 }
product:
  | product TIMES factor { Binary ("*", $1, $3) }
  | product DIVIDE factor { Binary ("/", $1, $3) }
  | factor { $1 }
factor:
  | LPAREN expr RPAREN { $2 }
  | NUMBER { Number $1 }
  | ID { Name $1 }
```

In [ ]:
printfn "%s" (File.ReadAllText "Lalr/Symbolic.fsyacc.output")

## Evaluate expressions

The parser returns syntax; the evaluator owns variable lookup and arithmetic. Undefined names retain the original `NaN` semantics. A statement returns a new immutable environment plus an optional displayed value.

In [ ]:
let rec evaluate (values: Map<string, float>) expression =
    match expression with
    | Number value -> value
    | Name name -> Map.tryFind name values |> Option.defaultValue Double.NaN
    | Binary (operator, left, right) ->
        let x, y = evaluate values left, evaluate values right
        match operator with
        | "+" -> x + y
        | "-" -> x - y
        | "*" -> x * y
        | "/" when y = 0.0 -> raise (DivideByZeroException "Division by zero")
        | "/" -> x / y
        | _ -> invalidArg "expression" operator
    | other -> invalidArg "expression" (sprintf "%A" other)

let execute values statement =
    match statement with
    | Assign (name, expression) -> Map.add name (evaluate values expression) values, None
    | Evaluate expression -> values, Some (evaluate values expression)

let calculate values text = Parse.symbolic text |> execute values

## Floating-point values

.NET supports IEEE floating-point values, including `NaN` and positive infinity. Arithmetic propagates an undefined value rather than inventing a variable value.

In [ ]:
Double.NaN, Double.PositiveInfinity, Double.PositiveInfinity - Double.PositiveInfinity
checkNotebook (Double.IsNaN (Double.PositiveInfinity - Double.PositiveInfinity))
checkNotebook (1.0 / Double.PositiveInfinity = 0.0)

## A reproducible session

Each line ends with a semicolon. Unlike an input loop, these cells can run unattended; the optional console loop below uses the same `calculate` function.

In [ ]:
let session = ["x := 2;"; "y := 3.5;"; "x + 2*y;"; "undefined;"; "zero := 0.25e2;"]
let values, outputs =
    session |> List.fold (fun (environment, outputs) text ->
        let next, value = calculate environment text
        next, outputs @ [value]) (Map.empty, [])
checkNotebook (values["x"] = 2.0 && values["y"] = 3.5 && values["zero"] = 25.0)
checkNotebook (outputs[2] = Some 9.0)
checkNotebook (outputs[3] |> Option.exists Double.IsNaN)
values, outputs

In [ ]:
let consoleSession () =
    let rec loop values =
        printf "calc> "
        match Console.ReadLine() with
        | null | "" -> values
        | text ->
            try
                let next, result = calculate values text
                result |> Option.iter (printfn "%g")
                loop next
            with error ->
                printfn "%s" error.Message
                loop values
    loop Map.empty
// Call consoleSession() explicitly in F# Interactive to read from the console.

In [ ]:
let invalid = try calculate Map.empty "1 + ;" |> ignore; false with _ -> true
checkNotebook invalid
for source in ["x<y;"; "x>y;"; "007;"] do
    let rejected = try calculate Map.empty source |> ignore; false with _ -> true
    checkNotebook rejected

let divisionRejected = try calculate Map.empty "1/0;" |> ignore; false with :? DivideByZeroException -> true
checkNotebook divisionRejected
